In [92]:
import requests
import pandas as pd
from datetime import date, timedelta
import time
import os
from bs4 import BeautifulSoup
import numpy as np

In [103]:
BASE_URL = "https://www.airkorea.or.kr/web/pmRelaySub"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/129.0 Safari/537.36"
    )
}
DISTRICTS = [
    "053","054","055","061","062","063","064",
]

In [94]:
def fetch_airkorea_day(target_date: date, district: str, item_code="11008"): 
    """
    하루치 데이터를 DataFrame으로 반환. (테이블 없으면 None)
    """
    params = {
        "strDateDiv": "1",
        "searchDate": target_date.strftime("%Y-%m-%d"),
        "district": district,
        "itemCode": item_code,
        "searchDate_f": target_date.strftime("%Y%m"),
    }

    resp = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=600)

    if resp.status_code != 200:
        return None

    try:
        tables =  resp.text
    except:
        return None

    if not tables:
        return None
    
    return tables

In [95]:
def parse_pm_table(html: str, date_str: str, district: str) -> pd.DataFrame:
    """
    위에 붙여준 HTML처럼 생긴 문자열에서 미세먼지 테이블만 뽑아서 DataFrame으로 변환
    """
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", id="tablefix1")
    if table is None:
        raise ValueError("tablefix1 테이블을 찾지 못했습니다.")

    # pandas가 <th>를 헤더로 인식해서 바로 DataFrame으로 변환
    try:
        df = pd.read_html(str(table))[0]
    except Exception as e:
        return None
    # display(df)
    # 날짜/지역번호 컬럼 추가
    df.insert(0, "date", date_str)
    # df.insert(1, "district", district)

    # # '-' → NaN으로 치환
    # df = df.replace("-", np.nan)

    # # 시간 컬럼 숫자로 변환 (측정망/측정소명 제외)
    # hour_cols = df.columns[3:]
    # df[hour_cols] = df[hour_cols].apply(pd.to_numeric, errors="coerce")

    return df

In [96]:
def crawl_district(district: str, start: date, end: date, item_code="11008"):
    """
    특정 지역번호에 대해 start~end 데이터를 모두 수집.
    """
    all_df = []
    cur = start

    while cur <= end:
        print(f"[{district}] Fetching {cur} ...")
        tables = fetch_airkorea_day(cur, district, item_code=item_code)
        if tables is None:
            continue
        df = parse_pm_table(tables, cur.strftime("%Y-%m-%d"), district)

        if df is not None:
            all_df.append(df)

        time.sleep(0.5)  # 서버 보호

        cur += timedelta(days=1)

    if not all_df:
        print(f"[{district}] No data found.")
        return None

    return pd.concat(all_df, ignore_index=True)

In [97]:
def crawl_all_districts(start: date, end: date, item_code="11008"):
    os.makedirs("airkorea_result", exist_ok=True)

    for district in DISTRICTS:
        print(f"\n==== 지역번호 {district} 수집 시작 ====")
        df = crawl_district(district, start, end, item_code=item_code)

        if df is None:
            print(f"[{district}] 데이터 없음 → SKIP")
            continue

        file_path = f"airkorea_result/airkorea_25_{district}_{start}_{end}.csv"
        df.to_csv(file_path, index=False, encoding="utf-8-sig")
        print(f"[{district}] 저장 완료 → {file_path}")

In [104]:
start_date = date(2024, 1, 1)
end_date   = date(2025, 11, 26)


crawl_all_districts(start_date, end_date)


==== 지역번호 053 수집 시작 ====
[053] Fetching 2024-01-01 ...


C:\Users\chuyk\AppData\Local\Temp\ipykernel_13264\1406786127.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


[053] Fetching 2024-01-02 ...


C:\Users\chuyk\AppData\Local\Temp\ipykernel_13264\1406786127.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


[053] Fetching 2024-01-03 ...


...

[064] Fetching 2025-11-25 ...


C:\Users\chuyk\AppData\Local\Temp\ipykernel_13264\1406786127.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


[064] Fetching 2025-11-26 ...


C:\Users\chuyk\AppData\Local\Temp\ipykernel_13264\1406786127.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


[064] 저장 완료 → airkorea_result/airkorea_25_064_2024-01-01_2025-11-26.csv


In [65]:
pip install html5lib


   -------------------- ------------------- 1/2 [html5lib]
   -------------------- ------------------- 1/2 [html5lib]
   -------------------- ------------------- 1/2 [html5lib]
   -------------------- ------------------- 1/2 [html5lib]
   ---------------------------------------- 2/2 [html5lib]

Note: you may need to restart the kernel to use updated packages.
